## Polyglot Notebook — Cell Identity & Kernel-Routing UI

**Problem.** The notebook is polyglot end-to-end: every code cell carries `metadata.spur.code_type ∈ {python, javascript, rust, go}` and routes to a distinct kernel (`python3 / deno / evcxr / gonb`), with `spur` → the AI backend. But the frontend renders it monolingual — the editor hardcodes Python highlighting (`CellInput.tsx:83`), no language is ever shown, and there's no switcher. The only per-cell identity that exists is the `✦ AI` chrome.

**Design (full system, approved).** Make language/kernel a first-class cell identity, with the `✦ AI` cell as one entry in the same system:

1. **Language chip** — glyph + accent per kernel (Py · JS · Rs · Go · ✦ AI), in the header zone we already use for `✦ AI`.
2. **Chip *is* the switcher** — click → kernel/type menu, replacing the binary code↔markdown toggle.
3. **Per-`code_type` highlighting** — drive CodeMirror off `codeType`, not hardcoded Python.

Board below renders the system across all five cell types in one document.

In [2]:
# open-design artifact — polyglot cell identity system
from IPython.display import HTML

HTML(r"""
<div style="background:#f1f1f3;padding:28px;font-family:Inter,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;color:#1f2933;line-height:1.45;">
<style>
  .pg-wrap{max-width:1080px;margin:0 auto;}
  .pg-mono{font-family:'Fira Code','SF Mono',ui-monospace,Menlo,monospace;}
  .pg-h{font-size:22px;font-weight:700;letter-spacing:-.01em;margin:0 0 2px;}
  .pg-sub{font-size:13px;color:#6b7280;margin:0 0 22px;}
  .pg-card{background:#fff;border:1px solid #e5e7eb;border-radius:10px;box-shadow:0 1px 2px rgba(16,24,40,.04);}
  .pg-sec{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;color:#9aa3af;margin:26px 2px 10px;}

  /* language token legend */
  .pg-legend{display:grid;grid-template-columns:repeat(5,1fr);gap:10px;}
  .pg-tok{border:1px solid #e5e7eb;border-radius:9px;padding:12px;background:#fff;}
  .pg-glyph{display:inline-flex;align-items:center;justify-content:center;width:26px;height:26px;border-radius:6px;font-weight:700;font-size:11px;}
  .pg-tokname{font-weight:700;font-size:13px;margin-top:9px;}
  .pg-tokspec{font-size:11px;color:#9aa3af;margin-top:1px;}

  /* notebook mock */
  .pg-nb{padding:6px 0;}
  .pg-cell{position:relative;padding:0 18px 0 0;}
  .pg-hr{border:0;border-top:1px solid #eceef1;margin:0;}
  .pg-gutter{position:absolute;left:0;top:15px;width:57px;text-align:center;font-size:10.5px;line-height:20px;}
  .pg-acctbar{position:absolute;left:0;top:14px;width:3px;height:calc(100% - 24px);border-radius:2px;}
  .pg-chiprow{display:flex;align-items:center;gap:8px;padding:11px 0 6px 57px;}
  .pg-chip{display:inline-flex;align-items:center;gap:6px;border-radius:6px;padding:2px 8px 2px 4px;font-size:11px;font-weight:600;border:1px solid;cursor:pointer;}
  .pg-chip .cg{width:18px;height:18px;border-radius:4px;display:inline-flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;}
  .pg-caret{color:#b6bcc6;font-size:9px;margin-left:1px;}
  .pg-pill{border-radius:5px;padding:1px 7px;font-size:9px;font-weight:600;border:1px solid;}
  .pg-code{padding:4px 18px 16px 57px;font-size:13px;white-space:pre;overflow:hidden;}
  .pg-out{padding:6px 18px 14px 57px;font-size:12px;color:#374151;}
  .pg-outtable{border-collapse:collapse;font-size:11.5px;}
  .pg-outtable td,.pg-outtable th{border:1px solid #e8eaed;padding:2px 9px;text-align:right;}
  .pg-outtable th{background:#fafbfc;color:#6b7280;font-weight:600;}
  .pg-md{padding:10px 18px 14px 57px;}
  .pg-md h3{font-size:16px;margin:2px 0 4px;font-weight:700;}
  .pg-md p{margin:0;color:#4b5563;font-size:13px;}

  /* syntax tokens */
  .kw{color:#7c3aed;} .str{color:#b91c1c;} .num{color:#047857;} .com{color:#9aa3af;font-style:italic;}
  .def{color:#1d4ed8;} .fn{color:#0369a1;} .ty{color:#b45309;}

  /* switcher menu */
  .pg-menu{position:absolute;z-index:5;left:57px;top:34px;width:226px;background:#fff;border:1px solid #e5e7eb;border-radius:9px;box-shadow:0 12px 28px rgba(16,24,40,.16);padding:5px;}
  .pg-mi{display:flex;align-items:center;gap:9px;padding:6px 8px;border-radius:6px;font-size:12px;}
  .pg-mi.sel{background:#f5f3ff;}
  .pg-mi .ck{margin-left:auto;color:#7c3aed;font-weight:700;}
  .pg-mdiv{height:1px;background:#eef0f2;margin:5px 4px;}
  .pg-note{display:flex;gap:10px;align-items:flex-start;font-size:12px;color:#4b5563;padding:9px 12px;border:1px dashed #d8dce1;border-radius:8px;background:#fbfbfc;}
  .pg-cap{position:absolute;font-size:10.5px;color:#9aa3af;font-weight:600;}
</style>

<div class="pg-wrap">
  <div class="pg-h">Polyglot Notebook · Cell Identity &amp; Kernel Routing</div>
  <div class="pg-sub">One document, five routing targets. Each cell declares its language; the kernel layer routes it. The chip makes that visible and switchable.</div>

  <!-- LEGEND -->
  <div class="pg-sec">The five routing targets · <span class="pg-mono" style="text-transform:none;letter-spacing:0;">code_type → kernelspec</span></div>
  <div class="pg-legend">
    <div class="pg-tok"><span class="pg-glyph" style="background:#eaf2fb;color:#2c5e8a;">Py</span><div class="pg-tokname">Python</div><div class="pg-tokspec pg-mono">python3</div></div>
    <div class="pg-tok"><span class="pg-glyph" style="background:#fcf7e8;color:#8a6d00;">JS</span><div class="pg-tokname">JavaScript</div><div class="pg-tokspec pg-mono">deno</div></div>
    <div class="pg-tok"><span class="pg-glyph" style="background:#fbe9e4;color:#b23a22;">Rs</span><div class="pg-tokname">Rust</div><div class="pg-tokspec pg-mono">evcxr</div></div>
    <div class="pg-tok"><span class="pg-glyph" style="background:#e5f6fb;color:#0a7e9e;">Go</span><div class="pg-tokname">Go</div><div class="pg-tokspec pg-mono">gonb</div></div>
    <div class="pg-tok" style="border-color:#ddd6fe;"><span class="pg-glyph" style="background:#f5f3ff;color:#6d28d9;">✦</span><div class="pg-tokname" style="color:#6d28d9;">AI Agent</div><div class="pg-tokspec pg-mono">spur</div></div>
  </div>

  <!-- NOTEBOOK MOCK -->
  <div class="pg-sec">In situ · five cell types coexisting in one notebook</div>
  <div class="pg-card pg-nb">

    <!-- markdown cell -->
    <div class="pg-cell"><hr class="pg-hr"/>
      <div class="pg-md"><h3>Quarterly revenue model</h3><p>Mixed-language pipeline: load in Python, reshape in Rust, summarize with the agent.</p></div>
    </div>

    <!-- python cell -->
    <div class="pg-cell"><hr class="pg-hr"/>
      <span class="pg-acctbar" style="background:#3776AB;"></span>
      <div class="pg-gutter pg-mono" style="color:#9aa3af;">[1]</div>
      <div class="pg-chiprow">
        <span class="pg-chip" style="border-color:#cfe0f1;background:#fff;color:#2c5e8a;"><span class="cg" style="background:#eaf2fb;">Py</span>Python<span class="pg-caret">▾</span></span>
      </div>
      <div class="pg-code pg-mono"><span class="kw">import</span> pandas <span class="kw">as</span> pd
df = pd.<span class="fn">read_parquet</span>(<span class="str">"rev.parquet"</span>)
df.<span class="fn">head</span>(<span class="num">2</span>)</div>
      <div class="pg-out">
        <table class="pg-outtable"><tr><th>q</th><th>region</th><th>rev</th></tr>
        <tr><td>Q1</td><td>EMEA</td><td>1.84M</td></tr><tr><td>Q1</td><td>APAC</td><td>2.10M</td></tr></table>
      </div>
    </div>

    <!-- rust cell -->
    <div class="pg-cell"><hr class="pg-hr"/>
      <span class="pg-acctbar" style="background:#CE422B;"></span>
      <div class="pg-gutter pg-mono" style="color:#9aa3af;">[2]</div>
      <div class="pg-chiprow">
        <span class="pg-chip" style="border-color:#e8b9ac;background:#fff;color:#b23a22;"><span class="cg" style="background:#fbe9e4;">Rs</span>Rust<span class="pg-caret">▾</span></span>
      </div>
      <div class="pg-code pg-mono"><span class="kw">let</span> growth: <span class="ty">Vec</span>&lt;<span class="ty">f64</span>&gt; = rev.<span class="fn">windows</span>(<span class="num">2</span>)
    .<span class="fn">map</span>(|w| (w[<span class="num">1</span>] - w[<span class="num">0</span>]) / w[<span class="num">0</span>])
    .<span class="fn">collect</span>();</div>
      <div class="pg-out pg-mono" style="color:#6b7280;">[0.142, 0.087, 0.203]</div>
    </div>

    <!-- go cell -->
    <div class="pg-cell"><hr class="pg-hr"/>
      <span class="pg-acctbar" style="background:#00ADD8;"></span>
      <div class="pg-gutter pg-mono" style="color:#9aa3af;">[3]</div>
      <div class="pg-chiprow">
        <span class="pg-chip" style="border-color:#a8deec;background:#fff;color:#0a7e9e;"><span class="cg" style="background:#e5f6fb;">Go</span>Go<span class="pg-caret">▾</span></span>
      </div>
      <div class="pg-code pg-mono"><span class="kw">func</span> <span class="fn">forecast</span>(g []<span class="ty">float64</span>) <span class="ty">float64</span> {
    <span class="kw">return</span> g[<span class="kw">len</span>(g)-<span class="num">1</span>] * <span class="num">1.15</span>
}</div>
    </div>

    <!-- ai cell (manual) with switcher OPEN -->
    <div class="pg-cell"><hr class="pg-hr"/>
      <span class="pg-acctbar" style="background:#7c3aed;"></span>
      <div class="pg-gutter pg-mono" style="color:#6d28d9;">✦[4]</div>
      <div class="pg-chiprow">
        <span class="pg-chip" style="border-color:#ddd6fe;background:#f5f3ff;color:#6d28d9;"><span class="cg" style="background:#fff;">✦</span>AI Agent<span class="pg-caret">▾</span></span>
        <span class="pg-pill" style="border-color:#d1d5db;background:#fff;color:#6b7280;">manual</span>
      </div>
      <span class="pg-cap" style="right:24px;top:14px;">▲ chip is the switcher</span>
      <div class="pg-menu">
        <div class="pg-mi"><span class="pg-glyph" style="width:18px;height:18px;font-size:9px;background:#eaf2fb;color:#2c5e8a;">Py</span>Python</div>
        <div class="pg-mi"><span class="pg-glyph" style="width:18px;height:18px;font-size:9px;background:#fcf7e8;color:#8a6d00;">JS</span>JavaScript</div>
        <div class="pg-mi"><span class="pg-glyph" style="width:18px;height:18px;font-size:9px;background:#fbe9e4;color:#b23a22;">Rs</span>Rust</div>
        <div class="pg-mi"><span class="pg-glyph" style="width:18px;height:18px;font-size:9px;background:#e5f6fb;color:#0a7e9e;">Go</span>Go</div>
        <div class="pg-mi sel"><span class="pg-glyph" style="width:18px;height:18px;font-size:9px;background:#f5f3ff;color:#6d28d9;">✦</span>AI Agent<span class="ck">✓</span></div>
        <div class="pg-mdiv"></div>
        <div class="pg-mi"><span style="width:18px;text-align:center;color:#9aa3af;">M</span>Markdown</div>
        <div class="pg-mi"><span style="width:18px;text-align:center;color:#9aa3af;">R</span>Raw</div>
      </div>
      <div class="pg-code" style="font-style:italic;color:#4b5563;white-space:normal;padding-top:42px;">“Summarize the revenue trend across regions and flag any quarter below 5% growth.”</div>
      <div class="pg-out" style="border-left:2px solid #ede9fe;margin-left:57px;padding-left:12px;color:#374151;">APAC leads at 20.3% QoQ; EMEA steady. <b>No quarter fell below 5%.</b></div>
    </div>

  </div>

  <!-- STATES -->
  <div class="pg-sec">Cell states · the chip is consistent, the accent does the identity work</div>
  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;">
    <div class="pg-card" style="padding:14px;">
      <div style="font-size:11px;color:#9aa3af;font-weight:600;margin-bottom:9px;">RUNNING</div>
      <div style="display:flex;align-items:center;gap:8px;">
        <span class="pg-mono" style="color:#9aa3af;font-size:11px;">[*]</span>
        <span class="pg-chip" style="border-color:#cfe0f1;color:#2c5e8a;"><span class="cg" style="background:#eaf2fb;">Py</span>Python</span>
        <span style="width:7px;height:7px;border-radius:50%;background:#3776AB;animation:none;"></span>
      </div>
    </div>
    <div class="pg-card" style="padding:14px;">
      <div style="font-size:11px;color:#9aa3af;font-weight:600;margin-bottom:9px;">AI · LIVE</div>
      <div style="display:flex;align-items:center;gap:8px;">
        <span class="pg-mono" style="color:#6d28d9;font-size:11px;">✦[*]</span>
        <span class="pg-chip" style="border-color:#ddd6fe;background:#f5f3ff;color:#6d28d9;"><span class="cg" style="background:#fff;">✦</span>AI Agent</span>
        <span class="pg-pill" style="border-color:#7c3aed;background:#7c3aed;color:#fff;">● LIVE</span>
      </div>
    </div>
    <div class="pg-card" style="padding:14px;">
      <div style="font-size:11px;color:#9aa3af;font-weight:600;margin-bottom:9px;">ERROR</div>
      <div style="display:flex;align-items:center;gap:8px;">
        <span class="pg-mono" style="color:#dc2626;font-size:11px;">[5]</span>
        <span class="pg-chip" style="border-color:#e8b9ac;color:#b23a22;"><span class="cg" style="background:#fbe9e4;">Rs</span>Rust</span>
        <span style="color:#dc2626;font-size:13px;">✕</span>
      </div>
    </div>
  </div>

  <!-- NOTES -->
  <div class="pg-sec">Scope boundary</div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;">
    <div class="pg-note"><span style="color:#16a34a;font-weight:700;">FE</span><div><b>Frontend-derivable now</b> — chip identity, accent bar, gutter glyph, switcher menu, per-language CodeMirror highlighting (add <span class="pg-mono">lang-rust</span>, <span class="pg-mono">lang-javascript</span>; Go via stream-language), and writing <span class="pg-mono">code_type</span> on switch. Go also needs adding to the <span class="pg-mono">CodeType</span> binding.</div></div>
    <div class="pg-note"><span style="color:#b45309;font-weight:700;">BE</span><div><b>Backend-gated</b> — AI <span class="pg-mono">● LIVE</span> auto-run cascade &amp; <span class="pg-mono">ai_live</span> persistence (bd-1bpb), agent name / usage / cached output. The chip shows the control; the wiring lands with the backend-surface epic.</div></div>
  </div>
</div></div>
""")


q,region,rev
Q1,EMEA,1.84M
Q1,APAC,2.10M
